### Chain Using LangGraph
In this section we will see how we can build a simple chain using LangGrap that uses 4 important concepts

- How to use chat messages as our graph state
- How to use chat models in graph nodes
- How to bind tools to our LLM in chat models
- How to execute the tools call in our graph nodes

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import os

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

### How to use chat messages as our graph state

#### Messages

We can use messages which can be used to capture different roles within a conversation. LangChain has various message types including HumanMessage, AIMessage, SystemMessage and ToolMessage. These represent a message from the user, from chat model, for the chat model to instruct behavior, and from tool call.

Every message have these important components.

- content - content of message
- name - specify the name of auditor
- response_metadata - optionally, a dict of metadata (eg- often populated by model provider for AIMessages)



In [2]:
from langchain_core.messages import AIMessage, HumanMessage
from pprint import pprint

messages= [ AIMessage(content=f"Please tell me how can I help", name="LLMModel") ]
messages.append(HumanMessage(content=f"I want to learn coding", name="Deepak"))
messages.append(AIMessage(content=f"Sure, I can help you with that. What programming language are you interested in?", name="LLMModel"))
messages.append(HumanMessage(content=f"I want to learn Python Programming Language", name="Deepak"))

for message in messages:
    message.pretty_print()

================================== Ai Message ==================================
Name: LLMModel

Please tell me how can I help
================================ Human Message =================================
Name: Deepak

I want to learn coding
================================== Ai Message ==================================
Name: LLMModel

Sure, I can help you with that. What programming language are you interested in?
================================ Human Message =================================
Name: Deepak

I want to learn Python Programming Language


Chat model

In [5]:
from langchain_groq import ChatGroq
llm=ChatGroq(model="llama-3.3-70b-versatile")
response=llm.invoke(messages)

In [6]:
response.pretty_print()

================================== Ai Message ==================================

Python is a popular and versatile language, widely used in various fields such as web development, data analysis, artificial intelligence, and more. Here's a step-by-step guide to help you get started:

**Setting up Python:**

1. **Download and Install Python**: Go to the official Python website ([www.python.org](http://www.python.org)) and download the latest version of Python for your operating system (Windows, macOS, or Linux).
2. **Choose a Text Editor or IDE**: A text editor or Integrated Development Environment (IDE) is where you'll write your Python code. Popular choices include:
	* PyCharm
	* Visual Studio Code (VS Code)
	* Sublime Text
	* Atom
	* IDLE (comes bundled with Python)
3. **Familiarize yourself with the interface**: Once you've installed Python and chosen a text editor or IDE, take some time to explore the interface and get comfortable with it.

**Learning Resources:**

1. **Official Py

In [7]:
response.response_metadata

{'token_usage': {'completion_tokens': 619,
  'prompt_tokens': 86,
  'total_tokens': 705,
  'completion_time': 2.006758767,
  'completion_tokens_details': None,
  'prompt_time': 0.00657492,
  'prompt_tokens_details': None,
  'queue_time': 0.163595807,
  'total_time': 2.013333687},
 'model_name': 'llama-3.3-70b-versatile',
 'system_fingerprint': 'fp_3272ea2d91',
 'service_tier': 'on_demand',
 'finish_reason': 'stop',
 'logprobs': None,
 'model_provider': 'groq'}

### Tools
tools can be integrated with the LLM modles to interact with internal systems. External systems can be API's, third party tools. 

whenever a query is asked the model can be choose to call the tool and this query is based on the natural language input and this will return an output that matches the tool's schema

In [8]:
def add(a:int, b:int)->int:
    """ Add a and b
    Args:
        a (int): first number
        b (int): second number
    Returns:
        int
    
    """
    return a+b

In [9]:
llm

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x11dc3b620>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x11dc39220>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [13]:
### binding tools with llm
llm_with_tools = llm.bind_tools([add])

tool_call = llm_with_tools.invoke([HumanMessage(content=f"What is the sum of 2 and 3?", name="Deepak")])

In [15]:
tool_call.tool_calls

[{'name': 'add',
  'args': {'a': 2, 'b': 3},
  'id': '3zxjnwvpf',
  'type': 'tool_call'}]

### using messages as state

In [17]:
from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage

class State(TypedDict):
    message: list[AnyMessage]



In [24]:
## Reducer

from langgraph.graph.message import add_messages
from typing import Annotated

class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    


In [21]:
### Reducer with add messages

initial_message = [AIMessage(content=f"Please tell me how can I help", name="LLMModel")]
initial_message.append(HumanMessage(content=f"I want to learn coding", name="Deepak"))
initial_message 

[AIMessage(content='Please tell me how can I help', additional_kwargs={}, response_metadata={}, name='LLMModel', tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='I want to learn coding', additional_kwargs={}, response_metadata={}, name='Deepak')]

In [22]:
ai_message = AIMessage(content=f"Sure, I can help you with that. What programming language are you interested in?", name="LLMModel")
ai_message


AIMessage(content='Sure, I can help you with that. What programming language are you interested in?', additional_kwargs={}, response_metadata={}, name='LLMModel', tool_calls=[], invalid_tool_calls=[])

In [23]:
### Reducers add_messages is to append messages instead of overriding
add_messages(initial_message, ai_message)

[AIMessage(content='Please tell me how can I help', additional_kwargs={}, response_metadata={}, name='LLMModel', id='42953082-3c85-4b4e-abf5-284a1c54c9c8', tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='I want to learn coding', additional_kwargs={}, response_metadata={}, name='Deepak', id='1ecfec7c-b56b-4770-9437-32399f54ee7a'),
 AIMessage(content='Sure, I can help you with that. What programming language are you interested in?', additional_kwargs={}, response_metadata={}, name='LLMModel', id='9d323f82-7610-40e4-a7f9-2f7e5d72782b', tool_calls=[], invalid_tool_calls=[])]

In [26]:
## chatbot node functionality

def llm_tool(state: State): 
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

In [ ]:
from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END
builder = StateGraph()